# 🧠 Machine Learning Classification (HW4)
Полное решение с метриками и графиками

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_curve, auc

from sklearn.ensemble import GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier


In [ ]:
data = load_breast_cancer()
X = data.data
y = data.target
feature_names = data.feature_names

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
models = {
    "GradientBoosting": GradientBoostingClassifier(),
    "CatBoost": CatBoostClassifier(verbose=0),
    "AdaBoost": AdaBoostClassifier(),
    "ExtraTrees": ExtraTreesClassifier(),
    "QDA": QuadraticDiscriminantAnalysis(),
    "LightGBM": LGBMClassifier(),
    "KNN": KNeighborsClassifier(),
    "DecisionTree": DecisionTreeClassifier(),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    "Dummy": DummyClassifier(),
    "SVM": SVC(kernel='linear', probability=True)
}

In [ ]:
results = []
trained_models = {}

for name, model in models.items():
    cv_score = cross_val_score(model, X_train, y_train, cv=5, scoring='f1').mean()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    trained_models[name] = model

    results.append({
        "Model": name,
        "CV_F1": cv_score,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred)
    })

df_results = pd.DataFrame(results).sort_values(by="F1", ascending=False)
df_results

In [ ]:
df_results.set_index("Model")[["Accuracy","Precision","Recall","F1"]].plot(kind="bar", figsize=(12,6))
plt.title("Model Comparison")
plt.show()

In [ ]:
plt.figure(figsize=(10,7))
for name, model in trained_models.items():
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:,1]
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        plt.plot(fpr, tpr, label=name)
plt.legend()
plt.title("ROC Curves")
plt.show()

In [ ]:
best_model_name = df_results.iloc[0]["Model"]
best_model = trained_models[best_model_name]
print("Best model:", best_model_name)

In [ ]:
y_pred = best_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

sns.heatmap(cm, annot=True, fmt="d")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
model = trained_models["ExtraTrees"]
importances = model.feature_importances_
indices = np.argsort(importances)[-10:]

plt.barh(range(len(indices)), importances[indices])
plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
plt.title("Top Features")
plt.show()